In [1]:
# Multiple ways to get and work with home directory
import os
from pathlib import Path
import spacy
import scispacy
import pandas as pd
import math
import numpy as np
import requests
import re
import time
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity
from torch.nn.functional import cosine_similarity as torch_cosine_similarity
from functools import lru_cache
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import rankdata

# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
#print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Datasets"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")

Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi
Current working directory: /Users/naveendeevi/Datasets


In [35]:
def get_openalex_references_and_citations(work_id):
    """
    Given an OpenAlex Work ID, return a dictionary of its references and citations.

    Args:
        work_id (str): OpenAlex Work ID (e.g., "W4406696697")

    Returns:
        dict: {
            'id': str,
            'title': str,
            'referenced_works': List[str],
            'cited_by_works': List[str]
        }
    """
    base_url = f"https://api.openalex.org/works/{work_id}"
    response = requests.get(base_url)

    if response.status_code != 200:
        raise Exception(f"Error fetching data for {work_id}: {response.status_code}")

    data = response.json()
    referenced_works = data.get("referenced_works", [])
    
    # Get citing works via filter
    citations_url = f"https://api.openalex.org/works?filter=cites:{work_id}&per-page=200"
    citations_response = requests.get(citations_url)

    if citations_response.status_code != 200:
        raise Exception(f"Error fetching citations for {work_id}: {citations_response.status_code}")
    
    citations_data = citations_response.json()
    citing_ids = [item["id"].split("/")[-1] for item in citations_data.get("results", [])]

    return {
        "id": data.get("id", "").split("/")[-1],
        "title": data.get("title", ""),
        "referenced_works": referenced_works,
        "cited_by_works": citing_ids
    }


## Function that populates the Pool Collection DF. 
def create_openalex_entry_df(identifier):
    """
    Given an OpenAlex ID or link, return a DataFrame with metadata:
    ID, Title, API Link, Citation/Reference, and Author Names.

    Args:
        identifier (str): OpenAlex Work ID (e.g., 'W2046695653') or URL (e.g., 'https://openalex.org/W2049189688')

    Returns:
        pd.DataFrame: With columns ['ID', 'Title', 'API link', 'Citation or Reference', 'Authors']
    """
    # Determine if input is a link or ID
    if identifier.startswith("https://openalex.org/"):
        work_id = identifier.split("/")[-1]
        label = "Reference"
    else:
        work_id = identifier
        label = "Citation"

    # Build API link
    api_link = f"https://api.openalex.org/works/{work_id}"

    # Fetch paper metadata
    response = requests.get(api_link)
    if response.status_code != 200:
        raise Exception(f"Failed to retrieve OpenAlex data: {response.status_code}")

    data = response.json()

    title = data.get("title", "")
    authorships = data.get("authorships", [])

    # Extract author names
    author_names = [auth["author"]["display_name"] for auth in authorships if "author" in auth]
    authors_str = ", ".join(author_names) if author_names else ""

    # Assemble row
    row = {
        "ID": work_id,
        "Title": title,
        "API link": api_link,
        "Citation or Reference": label,
        "Authors": authors_str
    }

    return pd.DataFrame([row])

## Similarity scores SentenceBERT

# Corpus-wide SBERT Similarity (One-to-Many)
def sbert_group_similarity(df, ref_col, cmp_col, score_col='SBERT_group_similarity'):
    model = SentenceTransformer('all-MiniLM-L6-v2')

    df = df.copy()
    df['ref_group'] = (df[ref_col] != df[ref_col].shift()).cumsum()

    all_scores = {}

    for gid, group in df.groupby('ref_group'):
        ref_text = group[ref_col].iloc[0]

        # Skip entire group if reference text is missing or invalid
        if pd.isna(ref_text) or not isinstance(ref_text, str) or not ref_text.strip():
            for idx in group.index:
                all_scores[idx] = np.nan
            continue

        cmp_texts = group[cmp_col].tolist()
        cmp_indices = group.index.tolist()

        sim_scores = []
        for t in cmp_texts:
            if pd.isna(t) or not isinstance(t, str) or not t.strip():
                sim_scores.append(np.nan)
            else:
                emb_ref = model.encode(ref_text, convert_to_tensor=True)
                emb_cmp = model.encode(t, convert_to_tensor=True)
                score = util.cos_sim(emb_ref, emb_cmp).item()
                sim_scores.append(score)

        for i, idx in enumerate(cmp_indices):
            all_scores[idx] = sim_scores[i]

    df[score_col] = df.index.to_series().map(all_scores)
    return df.drop(columns=['ref_group'])

# Word cloud & Sentence similarity
def word_cloud_jaccard_similarity(word_list, sentence):
    """
    Compute Jaccard similarity between a list of words and a sentence.

    Args:
        word_list (list): A list of words (e.g., MDD_Word_Cloud)
        sentence (str): A sentence to compare with the word list

    Returns:
        float: Jaccard similarity score (0.0 to 1.0)
    """
    if not isinstance(sentence, str) or not sentence.strip():
        return 0.0

    # Convert both to sets
    set1 = set(w.lower() for w in word_list)
    set2 = set(sentence.lower().split())

    # Compute Jaccard index
    intersection = set1.intersection(set2)
    union = set1.union(set2)

    if not union:
        return 0.0

    return len(intersection) / len(union)

MDD_Word_Cloud = ["5-HT1A", "5-HT4R", "Agomelatine", "AMPA Receptor Activation", "AMPA antagonist", "Analgesics", "Anesthetics",
    "Antidepressant", "Antidepressants", "Antipsychotic", "Antipsychotics", "Ayahuasca", "BDI", "BDNF", "Beta-arrestin",
    "Biomarker", "Biomarkers", "Brexpiprazole", "Bupropion", "Citalopram", "Clomipramine", "CYP2C19", "CYP2D6",
    "Central Nervous System Depressants", "Cyclooxygenase Inhibitors", "Desvenlafaxine", "Dopamine Reuptake Inhibitor",
    "Depression", "Depress", "Depressive", "MDD", "Duloxetine", "EEG", "Escitalopram", "Estrogen Antagonists", 
    "Excitatory Amino Acid Agents", "Excitatory Amino Acid Antagonists",
    "Fluoxetine", "GABA Agents", "Gene", "GeneSight", "Genetic", "Genetics", "Genomics", "General Anesthetics",
    "Glutamate", "Glutamate antagonist", "HAMD", "HAM-D", "HDRS", "Hydroxynorketamine", "IGF-1", "Inventory (generic)",
    "Ketamine", "Leucine", "MADRS", "MDMA", "MTHFR", "Magnetic Resonance Spectroscopy", "Magnetoencephalography",
    "Metabotropic glutamate receptor antagonist", "Mirtazapine", "Montgomery-Åsberg Depression Rating Scale",
    "NEGR1", "NMDA antagonist", "NMDA receptor antagonist", "Norepinephrine Reuptake Inhibitor", "Opioid",
    "PACAP", "PHQ", "PHQ-8", "PHQ-9", "PRKCB1", "Paroxetine", "Patient Health Questionnaire", "Peripheral Nervous System Agents",
    "Psilocybin", "P2X7", "QEEG", "QIDS", "SERT", "SNRI", "SNRIs", "SSRI", "SSRIs", "SV2A", "Selective Serotonin Reuptake Inhibitor",
    "Sensory System Agents", "Serotonin Uptake Inhibitors", "Serotonin transporter gene polymorphism",
    "SPECT", "Trazodone", "Thyroid pathway genes", "Triiodothyronine", "Venlafaxine", "Vilazodone", "Vortioxetine"
]

def take_pool_df_approved_rejected_df(pool_collection, similarity_threshold=0.10):
    filter_sheet = pd.DataFrame(columns=["Seed_Title","reference_ID","reference_citation_title","API link","Authors","SBERT_score","word_cloud_score"])

    for ids,title in zip(pool_collection['Seed_ID'],pool_collection['Seed_Title']):
        citations_references = get_openalex_references_and_citations(ids)
        #citations_references = get_semanticscholar_references_and_citations(ids)
        
        for i in citations_references['referenced_works']:
            x = create_openalex_entry_df(i)
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [title,x.loc[0,'ID'],x.loc[0,'Title'],x.loc[0,'API link'],x.loc[0,'Authors']]
        for j in citations_references['cited_by_works']:
            y = create_openalex_entry_df(j)
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [title,y.loc[0,'ID'],y.loc[0,'Title'],y.loc[0,'API link'],y.loc[0,'Authors']]

    filter_sheet_V2 = sbert_group_similarity(filter_sheet,'Seed_Title','reference_citation_title', score_col = 'SBERT_score')
    min_val = filter_sheet_V2['SBERT_score'].min()
    max_val = filter_sheet_V2['SBERT_score'].max()
    filter_sheet_V2['SBERT_score_normalized'] = (filter_sheet_V2['SBERT_score'] - min_val) / (max_val - min_val)
    filter_sheet_V2.drop(columns=['SBERT_score'],inplace=True)
    filter_sheet_V2['word_cloud_score'] = filter_sheet_V2['reference_citation_title'].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))

    ########### UL 001 ###########################
    #filter_sheet_V2['SBERT_percentile'] = rankdata(filter_sheet_V2['SBERT_score_normalized'], method='average') / len(filter_sheet_V2)
    #filter_sheet_V2.drop(columns=['SBERT_score_normalized'], inplace=True)
    #filter_sheet_V2['title_Status'] = np.where(
    #    (filter_sheet_V2['SBERT_percentile'] >= similarity_threshold) | (filter_sheet_V2['word_cloud_score'] > 1.00),'collection_pool','rejection_pool')
    ########### End of UL 001 ######################
    
    filter_sheet_V2['title_Status'] = np.where((filter_sheet_V2['SBERT_score_normalized'] >= similarity_threshold) | (filter_sheet_V2['word_cloud_score'] > 0.00), 'collection_pool', 'rejection_pool')

    approved_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'collection_pool']
    approved_df = approved_df[['Seed_Title','reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    approved_df.columns = ['Seed_Title','Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']

    rejected_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'rejection_pool']
    rejected_df = rejected_df[['reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    rejected_df.columns = ['Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']

    return approved_df,rejected_df

def iteratable_approve_reject(jump_df, similarity_threshold=0.10):
    filter_sheet = pd.DataFrame(columns=["Seed_Title","reference_ID","reference_citation_title","API link","Authors","SBERT_score","word_cloud_score"])

    for ids, title, seed_title in zip(jump_df['Seed_ID'], jump_df['Seed_Title'], jump_df['Seed_Anchor_Title']):
        citations_references = get_openalex_references_and_citations(ids)
        #citations_references = get_semanticscholar_references_and_citations(ids) 
        for i in citations_references['referenced_works']:
            x = create_openalex_entry_df(i)
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [seed_title,x.loc[0,'ID'],x.loc[0,'Title'],x.loc[0,'API link'],x.loc[0,'Authors']]
        for j in citations_references['cited_by_works']:
            y = create_openalex_entry_df(j)
            filter_sheet.loc[len(filter_sheet),['Seed_Title','reference_ID','reference_citation_title','API link','Authors']] = [seed_title,y.loc[0,'ID'],y.loc[0,'Title'],y.loc[0,'API link'],y.loc[0,'Authors']]

    filter_sheet_V2 = sbert_group_similarity(filter_sheet,'Seed_Title','reference_citation_title', score_col = 'SBERT_score')
    min_val = filter_sheet_V2['SBERT_score'].min()
    max_val = filter_sheet_V2['SBERT_score'].max()
    filter_sheet_V2['SBERT_score_normalized'] = (filter_sheet_V2['SBERT_score'] - min_val) / (max_val - min_val)
    filter_sheet_V2.drop(columns=['SBERT_score'], inplace=True)
    filter_sheet_V2['word_cloud_score'] = filter_sheet_V2['reference_citation_title'].apply(lambda sentence: word_cloud_jaccard_similarity(MDD_Word_Cloud, sentence))

    ########### UL 001 ###########################
    #filter_sheet_V2['SBERT_percentile'] = rankdata(filter_sheet_V2['SBERT_score_normalized'], method='average') / len(filter_sheet_V2)
    #filter_sheet_V2.drop(columns=['SBERT_score_normalized'], inplace=True)
    #filter_sheet_V2['title_Status'] = np.where(
    #    (filter_sheet_V2['SBERT_percentile'] >= similarity_threshold) | (filter_sheet_V2['word_cloud_score'] > 1.00),'collection_pool','rejection_pool')
    ########### End of UL 001 ######################
    filter_sheet_V2['title_Status'] = np.where((filter_sheet_V2['SBERT_score_normalized'] >= similarity_threshold) | (filter_sheet_V2['word_cloud_score'] > 0.00), 'collection_pool', 'rejection_pool')

    approved_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'collection_pool']
    approved_df = approved_df[['Seed_Title','reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    approved_df.columns = ['Seed_Title','Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']

    rejected_df = filter_sheet_V2[filter_sheet_V2['title_Status'] == 'rejection_pool']
    rejected_df = rejected_df[['reference_ID', 'reference_citation_title', 'API link', 'Authors']].copy()
    rejected_df.columns = ['Linked_Paper_ID', 'Linked_Paper_Title', 'Linked Paper API link', 'Linked Paper Authors']

    return approved_df, rejected_df

In [37]:
# === Main Expansion ===
jump = 1
current_jump = 0
seed_list = ['W2114058973']

pool_collection = pd.DataFrame(columns=["Seed_ID", "Seed_Title", "Seed API link", "Seed Authors", "Seed or Citation or Reference"])
merged_df = pd.DataFrame()

while current_jump <= jump:
    start_time = time.time()

    if current_jump == 0:
        print(f"\nJump {current_jump} - Initializing seed metadata")
        for i in seed_list:
            new_df = asyncio.get_event_loop().run_until_complete(get_openalex_entries_async([i]))
            #print(i)
            row = new_df.iloc[0]
            pool_collection = pd.concat([pool_collection, pd.DataFrame([{
                'Seed_ID': row['id'].split('/')[-1],
                'Seed_Title': row['title'],
                'Seed API link': row['id'],
                'Seed Authors': ", ".join([a['author']['display_name'] for a in row['authorships']]),
                'Seed or Citation or Reference': 'Seed'
            }])], ignore_index=True)
        print(f"Jump {current_jump} - pool_collection: {pool_collection.shape}")

    elif current_jump == 1:
        print(f"\nJump {current_jump} - First hop expansion")
        approved, rejected = take_pool_df_approved_rejected_df(pool_collection, similarity_threshold=0.90)
        merged_df = pd.merge(pool_collection, approved, on='Seed_Title', how='inner')
        merged_df['Seed_Anchor_ID'] = merged_df['Seed_ID']
        merged_df['Seed_Anchor_Title'] = merged_df['Seed_Title']
        merged_df['Jump_Level'] = 1
        print(f"Jump {current_jump} - merged_df: {merged_df.shape}")

    elif current_jump > 1:
        print(f"\nJump {current_jump} - Recursive expansion")

        if merged_df.empty:
            print("No merged_df from previous step. Skipping.")
            break

        jump_df = merged_df[['Seed_Anchor_ID', 'Seed_Anchor_Title', 'Linked_Paper_ID', 'Linked_Paper_Title']]\
                    .drop_duplicates().copy()
        jump_df.columns = ['Seed_Anchor_ID', 'Seed_Anchor_Title', 'Seed_ID', 'Seed_Title']

        approved, rejected = iteratable_approve_reject(jump_df, similarity_threshold=0.90)

        if approved.empty:
            print(f"No new approved papers at jump {current_jump}")
            break

        merged_df_2 = pd.merge(jump_df, approved, on='Seed_Title', how='inner')
        merged_df_2['Jump_Level'] = current_jump

        if 'Linked_Paper_ID' in merged_df.columns:
            merged_df_2 = merged_df_2[~merged_df_2['Linked_Paper_ID'].isin(merged_df['Linked_Paper_ID'])]

        merged_df = pd.concat([merged_df, merged_df_2], ignore_index=True)
        print(f"Jump {current_jump} - merged_df updated: {merged_df.shape}")

    total_time = time.time() - start_time
    print(f"Jump {current_jump} - TOTAL time: {total_time:.2f}s")

    current_jump += 1



Jump 0 - Initializing seed metadata
Jump 0 - pool_collection: (1, 5)
Jump 0 - TOTAL time: 0.01s

Jump 1 - First hop expansion


/opt/anaconda3/envs/TransformersCleanNLPEnv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Jump 1 - merged_df: (168, 12)
Jump 1 - TOTAL time: 218.95s
